In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from torchinfo import summary
from of_transformer import OfTransformer
from net_utils import *
from data_utils import *

from sklearn.model_selection import train_test_split

import numpy as np
import uproot
import awkward as ak

Start by loading a single file for the MG sample using uproot. We will try to distinguish these events against themselves for now.

In [3]:
# Load data
f = uproot.open('../data/WithTracks_ZjetOmnifold_May19_MGPy8FxFxRew_syst_train.root')
tree = f['OmniTree']
tree.show(name_width=50)

name                                               | typename                 | interpretation                
---------------------------------------------------+--------------------------+-------------------------------
weight                                             | float                    | AsDtype('>f4')
pass190                                            | int32_t                  | AsDtype('>i4')
truth_pass190                                      | int32_t                  | AsDtype('>i4')
weight_mc                                          | float                    | AsDtype('>f4')
prw                                                | float                    | AsDtype('>f4')
pass190_syst_ID_Up                                 | int32_t                  | AsDtype('>i4')
pass190_syst_ID_Down                               | int32_t                  | AsDtype('>i4')
pass190_syst_MS_Up                                 | int32_t                  | AsDtype('>i4')
pass190_syst_MS_Do

Next we need to build torch tensors from the data. One open question is how to handle the muon kinematics, since they are distinct from the tracks. For now I will include the muon information in the same way as the tracks, but this should be changed in the future. Another open question is whether all of the data will fit in a A100s GPU memory. For now, I will just load whatever fits in my computer's RAM.

In [18]:
# Pass 190 flag
pass190 = ak.to_numpy(tree['pass190'].array())
print(pass190[:10])
print(np.sum(pass190) / len(pass190))

[1 1 1 1 1 0 0 1 1 0]
0.8144455536320278


In [21]:
# Muon information
m1_pt = ak.to_numpy(tree['pT_l1'].array())
m1_eta = ak.to_numpy(tree['eta_l1'].array())
m1_phi = ak.to_numpy(tree['phi_l1'].array())
m2_pt = ak.to_numpy(tree['pT_l2'].array())
m2_eta = ak.to_numpy(tree['eta_l2'].array())
m2_phi = ak.to_numpy(tree['phi_l2'].array())

m1_kinematics = np.stack([m1_pt, m1_eta, m1_phi], axis=1)
m2_kinematics = np.stack([m2_pt, m2_eta, m2_phi], axis=1)
muon_kinematics = np.stack([m1_kinematics, m2_kinematics], axis=2)

In [5]:
# Track information
track_pt = tree['pT_tracks'].array()
track_eta = tree['eta_tracks'].array()
track_phi = tree['phi_tracks'].array()

In [61]:
# Number of tracks information
ncs = ak.to_numpy(ak.num(track_pt, axis=1))
ncs = ncs[pass190 == 1]
print(ncs.shape)
no_track_idx = np.asarray(ncs == 0).nonzero()[0]

(1160150,)


In [62]:
pad_pt = pad_kinematics(track_pt, max_tracks=None)
pad_eta = pad_kinematics(track_eta, max_tracks=None)
pad_phi = pad_kinematics(track_phi, max_tracks=None)

In [63]:
track_kinematics = np.stack([pad_pt, pad_eta, pad_phi], axis=1)
print(track_kinematics.shape)

(1424466, 3, 292)


In [64]:
# Concatenate muon and track kinematics
kinematics = np.concatenate([muon_kinematics, track_kinematics], axis=2)
print(kinematics.shape)

# Filter kinematics by pass 190 flag
kinematics = kinematics[pass190 == 1,...]
print(kinematics.shape)

(1424466, 3, 294)
(1160150, 3, 294)


In [69]:
# Make one-hot encoding identifying whether the object is a muon or a track
is_muon = np.concatenate([np.ones((kinematics.shape[0], 2)), np.zeros((kinematics.shape[0], track_kinematics.shape[2]))], axis=1)
is_track = np.concatenate([np.zeros((kinematics.shape[0], 2)), np.ones((kinematics.shape[0], track_kinematics.shape[2]))], axis=1)
one_hot = np.stack([is_muon, is_track], axis=1)
print(one_hot.shape)


(1160150, 2, 294)


In [42]:
# Weights
weights = ak.to_numpy(tree['weight'].array())
weights /= np.mean(weights)
weights = np.expand_dims(weights[pass190 == 1], axis=1)
print(weights.shape)

(1160150, 1)


In [25]:
# Make a mask
mask = np.zeros_like(kinematics[:,0,:])
mask[kinematics[:,0,:] != 0] = 1
mask = np.expand_dims(mask, axis=1)
print(mask[0,0,:])
print(mask.shape)

[1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1.
 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0.]
(1160150, 1, 294)


In [26]:
# Construct a random vector of labels, these need to be float types to make the pytorch loss function happy
labels = np.random.randint(0, 2, size=(kinematics.shape[0], 1)).astype(np.float32)
print(labels.shape)

(1160150, 1)


In [43]:
# Make train test split
kinematics_train, kinematics_test, labels_train, labels_test, mask_train, mask_test, weights_train, weights_test = train_test_split(kinematics, labels, mask, weights, test_size=0.2, random_state=42)
print(kinematics_train.shape, kinematics_test.shape, labels_train.shape, labels_test.shape, mask_train.shape, mask_test.shape, weights_train.shape, weights_test.shape)

(928120, 3, 294) (232030, 3, 294) (928120, 1) (232030, 1) (928120, 1, 294) (232030, 1, 294) (928120, 1) (232030, 1)


In [44]:
# Convert to torch tensors
kinematics_train = torch.tensor(kinematics_train, dtype=torch.float32)
kinematics_test = torch.tensor(kinematics_test, dtype=torch.float32)
labels_train = torch.tensor(labels_train, dtype=torch.float32)
labels_test = torch.tensor(labels_test, dtype=torch.float32)
mask_train = torch.tensor(mask_train, dtype=torch.float32)
mask_test = torch.tensor(mask_test, dtype=torch.float32)
weights_train = torch.tensor(weights_train, dtype=torch.float32)
weights_test = torch.tensor(weights_test, dtype=torch.float32)

In [45]:
# Build a pytorch dataset
train_dataset = torch.utils.data.TensorDataset(kinematics_train, labels_train, mask_train, weights_train)
test_dataset = torch.utils.data.TensorDataset(kinematics_test, labels_test, mask_test, weights_test)

In [46]:
# Build pytorch data loaders
batch_size = 32
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

Data is ready. Next we need to build a model!

In [47]:
part = OfTransformer(
    3,
    num_classes=1,
    trim=True,
    fc_params=[(100, 0.0), (100, 0.0), (1, 0.5)],
    pair_embed_dims=None
)

In [48]:
summary(part, input_shape=kinematics_train.shape[1:])

Layer (type:depth-idx)                                       Param #
OfTransformer                                                128
├─SequenceTrimmer: 1-1                                       --
├─Embed: 1-2                                                 --
│    └─BatchNorm1d: 2-1                                      6
│    └─Sequential: 2-2                                       --
│    │    └─LayerNorm: 3-1                                   6
│    │    └─Linear: 3-2                                      512
│    │    └─GELU: 3-3                                        --
│    │    └─LayerNorm: 3-4                                   256
│    │    └─Linear: 3-5                                      66,048
│    │    └─GELU: 3-6                                        --
│    │    └─LayerNorm: 3-7                                   1,024
│    │    └─Linear: 3-8                                      65,664
│    │    └─GELU: 3-9                                        --
├─ModuleList: 1-3      

In [49]:
# Practice forward pass
first_event = kinematics_train[0:1,...]
first_mask = mask_train[0:1,...]
with torch.no_grad():
    out = part(first_event, mask=first_mask)
print(out)


tensor([[-0.4905]])


Now train the model. Define and optimizer and a loss function

In [53]:
criterion = torch.nn.BCEWithLogitsLoss(reduction='none')
optimizer = torch.optim.AdamW(part.parameters(), lr=1e-1)

And train for a few steps

In [54]:
n_minibatch = 10
part.train()

# Get minibatch from pytorch data loader
batch = next(iter(train_loader))

# Unpack batch 
batch_kinematics, batch_labels, batch_mask, batch_weights = batch

for step in range(n_minibatch):  # Just run a few minibatches to ensure we get good gradients

    # # Get minibatch from pytorch data loader
    # batch = next(iter(train_loader))

    # # Unpack batch 
    # batch_kinematics, batch_labels, batch_mask, batch_weights = batch

    # Zero gradients
    optimizer.zero_grad()

    # Forward pass, compute loss, backward pass, optimizer step
    out = part(batch_kinematics, mask=batch_mask)
    loss = criterion(out, batch_labels)
    loss = loss * batch_weights
    loss.mean().backward()
    optimizer.step()

    # Print stats
    print(f'Step {step} loss: {loss.mean().item():.3f}')

print('Finished Training')

Quantile maxlen:  tensor(104)
Step 0 loss: 0.670
Quantile maxlen:  tensor(98)
Step 1 loss: 0.661
Quantile maxlen:  tensor(116)
Step 2 loss: 0.655
Quantile maxlen:  tensor(116)
Step 3 loss: 0.650
Quantile maxlen:  tensor(99)
Step 4 loss: 0.648
Quantile maxlen:  tensor(116)
Step 5 loss: 0.648
Quantile maxlen:  tensor(98)
Step 6 loss: 0.649
Quantile maxlen:  tensor(116)
Step 7 loss: 0.650
Quantile maxlen:  tensor(98)
Step 8 loss: 0.651
Quantile maxlen:  tensor(109)
Step 9 loss: 0.652
Finished Training
